# 03 — Experiments

**Purpose:** Run TRF analysis for a selected experiment configuration.

## How to switch experiments

Change **only** the `EXPERIMENT` variable in the cell below.
Everything else stays the same.

Available experiments:
| Config | Regressors | Scientific question |
|---|---|---|
| `motion_state_v3` | motion energy + velocity + stasis + events | Current best model |
| `baseline_motion` | motion energy only | Does overall movement predict EEG? |
| `sign_onset_only` | ELAN sign onset | Do linguistic sign boundaries predict EEG? |

Results are saved to `results/{EXPERIMENT}_results.npz`.

In [ ]:
# ============================================================
# CHANGE THIS to switch experiments
EXPERIMENT = 'motion_state_v3'
# ============================================================

In [ ]:
import sys, os
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.chdir(ROOT)

import numpy as np
import glob
print(f'Running experiment: {EXPERIMENT}')

## Step 1 — Load everything

Load session config, EEG (with filter), motion data, ELAN annotations,
and the selected experiment config.

In [ ]:
from sign_language_trf.config import load_session_config, load_experiment_config
from sign_language_trf.preprocessing.eeg_loader import load_eeg
from sign_language_trf.preprocessing.motion_loader import load_motion
from sign_language_trf.preprocessing.markers_loader import load_markers
from sign_language_trf.preprocessing.elan_loader import load_elan
from sign_language_trf.preprocessing.raw_rokoko_loader import load_raw_rokoko
from sign_language_trf.preprocessing.synchronization import (
    extract_presentation_segments, extract_segment_eeg
)
from sign_language_trf.regressors.manager import (
    RegressorManager, register_elan_regressors, register_raw_regressors
)

cfg     = load_session_config('config/session.yaml')
exp_cfg = load_experiment_config(f'config/experiments/{EXPERIMENT}.yaml')
fs      = float(cfg.common_rate)

print('Loading EEG...')
eeg = load_eeg(cfg.eeg_path, cfg, bandpass=True, l_freq=0.1, h_freq=30.0, verbose=False)
segs  = extract_presentation_segments(eeg, onset_code=255, offset_code=254)
presentations = [extract_segment_eeg(eeg, seg) for seg in segs]
print(f'  EEG: {len(presentations)} presentations, {presentations[0].shape[1]} channels')

print('Loading motion + ELAN...')
motion  = load_motion(cfg.motion_path)
markers = load_markers(cfg.markers_path)
elan    = load_elan(glob.glob('sign_language_trf/data/elan/*.eaf')[0])

mgr = RegressorManager(motion, markers, target_fs=fs)
register_elan_regressors(elan)

raw_files = glob.glob('sign_language_trf/data/rokoko/*raw*.csv')
if raw_files:
    register_raw_regressors(load_raw_rokoko(raw_files[0]))

X, feat_names = mgr.get_matrix(exp_cfg.regressors)
n_trim = min(X.shape[0], min(P.shape[0] for P in presentations))
X = X[:n_trim]
presentations = [P[:n_trim] for P in presentations]

print(f'  X: {X.shape}  features: {feat_names}')
print('Ready.')

## Step 2 — Run cross-validation

We use **leave-one-presentation-out** cross-validation with 5 folds.
Each fold holds out one of the 5 video repeats as a test set
and trains on the remaining 4.

The model is **ridge regression** (L2 regularization) fit using `mTRFpy`.
α (regularization strength) = 10,000,000 — found via alpha sweep on this dataset.

**What we get:** Pearson r per EEG channel, averaged across folds.
Higher r = the model better predicts that channel's response to motion.

In [ ]:
from sign_language_trf.pipeline import run_experiment
import time

print(f'Running {EXPERIMENT} with leave-one-out CV (5 folds)...')
t0 = time.time()

results = run_experiment(
    experiment_name=EXPERIMENT,
    X=X,
    presentations=presentations,
    exp_cfg=exp_cfg,
    session_cfg=cfg,
    verbose=True,
)

print(f'\nDone in {time.time()-t0:.1f}s')

## Step 3 — Results summary

Channels are ranked by prediction accuracy (Pearson r).
Left frontal channels (F7, AF3) and temporal channels (T7, CP5) are
expected to show the highest r — these are language and motor cortex areas.

In [ ]:
kernels  = results['kernels']        # (n_channels, n_features, n_delays)
r_ch     = results['r_per_channel']  # (n_channels,)
ch_names = eeg.ch_names
order    = np.argsort(r_ch)[::-1]

print(f'Experiment: {EXPERIMENT}')
print(f'Features  : {feat_names}')
print(f'Best channel: {ch_names[order[0]]}  r = {r_ch[order[0]]:.4f}')
print()
print(f'{"Channel":<8}  {"r":>7}')
print('-' * 18)
for i in order:
    print(f'{ch_names[i]:<8}  {r_ch[i]:>7.4f}')

## Step 4 — Save results

Results are saved as a `.npz` file in `results/`.
The visualization notebook loads from here.

In [ ]:
from sign_language_trf.config import load_experiment_config

delays_s = results.get('delays_s', None)

os.makedirs('results', exist_ok=True)
save_path = f'results/{EXPERIMENT}_results.npz'
np.savez(
    save_path,
    kernels      = kernels,
    r_per_channel= r_ch,
    feat_names   = np.array(feat_names),
    ch_names     = np.array(ch_names),
    delays_s     = delays_s,
    experiment   = EXPERIMENT,
)
print(f'Saved → {save_path}')
print(f'Keys  : kernels {kernels.shape}, r_per_channel {r_ch.shape}')